# GPT-500M Training — Kaggle 2×T4 DDP Edition

**Architecture:** Decoder-only GPT, ~505M params (28 layers × 1152 embd × 16 heads)  
**Data:** `open-web-math/open-web-math` streamed from HuggingFace  
**Target:** Kaggle 2×T4 (2 × 15.6 GB VRAM) via DistributedDataParallel  

### How DDP works here
- `torchrun` spawns 2 worker processes (one per GPU) by running `train_ddp.py`  
- Each process owns one GPU and processes its own micro-batches independently  
- Gradients are averaged across both GPUs via NCCL all-reduce before each optimizer step  
- Effective batch: 2 GPUs × micro_batch=1 × grad_accum=128 × 1024 = **262,144 tokens/step**  
- Only rank 0 saves checkpoints and prints eval results  

### Checkpoint compatibility
Checkpoints are cross-compatible with the single-GPU Colab notebook.  
The DDP `module.` prefix is stripped on save, so `.pt` files load cleanly on either platform.

### VRAM fixes (T4 OOM)
- **8-bit AdamW** (`bitsandbytes`) — cuts optimizer state from ~4GB to ~1GB; falls back to fp32 AdamW if not installed
- **micro_batch 2→1, grad_accum 64→128** — same effective batch, lower per-step peak
- **SDPA backend pinned** to memory-efficient attention — T4 has no flash-attn-v2 kernel and can otherwise silently fall back to the much heavier "math" backend
- **`empty_cache()`** after optimizer-state load, after DDP wrap, and every 50 steps — fights allocator fragmentation over long runs
- **`max_split_size_mb:128`** added to `PYTORCH_CUDA_ALLOC_CONF`

### Checkpoint persistence — Kaggle Model registry push
Every checkpoint save now also pushes a copy to a Kaggle Model as a new version (async, non-blocking), so it survives independently of `/kaggle/working` and doesn't depend on committing/downloading a 5.7GB file from a live session. One-time setup is in Section 3 (Kaggle API secrets + `KAGGLE_MODEL_HANDLE`).

### Flow
1. Install → 2. Check GPUs → 3. Seed checkpoint + configure registry push → 4. Write script → 5. Verify → 6. Launch  
Re-running cell 6 resumes automatically from the latest checkpoint.

---
### Parameter accounting
```
Token embedding    :  50257 × 1152  =   57.9M  (tied with lm_head)
Position embedding :   1024 × 1152  =    1.2M
Per transformer block (×28):
  CausalSelfAttn  :  4 × 1152²     =    5.31M
  FeedForward     :  8 × 1152²     =   10.62M
  LayerNorms ×2   :  4 × 1152      =    4608
28 blocks total                     =  447.0M
Final LayerNorm                     =    2304
──────────────────────────────────────────────
TOTAL                               = ~505.1M
```


## 1. Install Dependencies

In [ ]:
!pip install -q datasets transformers tokenizers tiktoken bitsandbytes

## 2. Environment Check

In [ ]:
import torch

n_gpus = torch.cuda.device_count()
print(f"GPUs available : {n_gpus}")
for i in range(n_gpus):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} — {props.total_memory / 1e9:.1f} GB")
print(f"PyTorch        : {torch.__version__}")

assert n_gpus >= 2, (
    "This notebook requires 2 GPUs. "
    "Select 'GPU T4 x2' in Kaggle accelerator settings."
)

## 3. Configure Paths & Seed Checkpoint

In [ ]:
import os, shutil, glob

CKPT_DIR  = '/kaggle/working/GPT500M/Checkpoints'
CACHE_DIR = '/tmp/HF_Cache'   # ephemeral — fine for streaming, saves working space
SCRIPT    = '/kaggle/working/train_ddp.py'

os.makedirs(CKPT_DIR,  exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

# ── Seed checkpoint from Kaggle Model registry (first session only) ───────────
existing = glob.glob(os.path.join(CKPT_DIR, 'ckpt_*.pt'))
if not existing:
    uploaded = glob.glob('/kaggle/input/**/ckpt_*.pt', recursive=True)
    if uploaded:
        src = sorted(uploaded)[-1]
        dst = os.path.join(CKPT_DIR, os.path.basename(src))
        print(f"Seeding checkpoint: {src} -> {dst}")
        shutil.copy(src, dst)
        print("Done. Training will resume from this checkpoint.")
    else:
        print("No uploaded checkpoint found — will start from scratch.")
else:
    print(f"Checkpoint already present: {sorted(existing)[-1]}")

print(f"\nCheckpoint dir : {CKPT_DIR}")
print(f"HF cache dir   : {CACHE_DIR}")

# ── Kaggle Model registry push target (checkpoint persistence) ────────────────
# Every checkpoint save will also push a copy to this Kaggle Model as a new
# version, so it survives independently of /kaggle/working and this session.
#
# One-time setup before this will work:
#   1. Create the model container once, from your own machine or the Kaggle
#      UI:  kaggle models create -p <empty_dir_with_a_model-metadata.json>
#      (or just click "New Model" on kaggle.com/models and note the slug).
#   2. Add your Kaggle API credentials as Kaggle Notebook secrets named
#      KAGGLE_USERNAME and KAGGLE_KEY (Add-ons -> Secrets). These are the
#      same values from the kaggle.json you'd download from
#      kaggle.com/settings -> Create New Token.
#   3. Set KAGGLE_MODEL_HANDLE below to "<your-username>/<model-slug>/pytorch/checkpoints"
#      (the /pytorch/checkpoints part is the framework/variation slug Kaggle
#      expects — adjust if you named your variation something else).
KAGGLE_MODEL_HANDLE = "your-username/gpt500m/pytorch/checkpoints"  # <-- EDIT ME

try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    os.environ["KAGGLE_USERNAME"] = _secrets.get_secret("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"]      = _secrets.get_secret("KAGGLE_KEY")
    os.environ["KAGGLE_MODEL_HANDLE"] = KAGGLE_MODEL_HANDLE
    print(f"Kaggle API credentials loaded. Checkpoints will be pushed to: {KAGGLE_MODEL_HANDLE}")
except Exception as e:
    os.environ["KAGGLE_MODEL_HANDLE"] = ""
    print(f"WARNING: Kaggle API secrets not found ({e}).")
    print("Checkpoints will stay local only (no registry push) until you add")
    print("KAGGLE_USERNAME / KAGGLE_KEY as Notebook secrets.")


## 4. Write Training Script

The full training logic lives in `train_ddp.py`.  
`torchrun` will launch one copy per GPU — each knows its `LOCAL_RANK` via environment variables.

In [ ]:
import json as _json
_script = _json.loads("\"import contextlib\\nimport os, gc, glob, math, time, subprocess, tempfile, shutil, threading\\nfrom dataclasses import dataclass\\nfrom typing import Optional, Iterator\\n\\nimport torch\\nimport torch.nn as nn\\nimport torch.nn.functional as F\\nimport torch.utils.checkpoint as grad_ckpt\\nimport torch.distributed as dist\\nfrom torch.nn.parallel import DistributedDataParallel as DDP\\n\\n# \\u2500\\u2500 SDPA backend pin \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n# T4 (Turing, sm_75) has no flash-attention-v2 kernel. Left to its own\\n# devices, PyTorch's SDPA dispatcher can silently fall back to the naive\\n# \\\"math\\\" backend, which materializes a full TxT attention matrix per head\\n# and burns far more VRAM. Force the memory-efficient backend explicitly.\\ntry:\\n    from torch.nn.attention import sdpa_kernel, SDPBackend\\n    def _sdpa_ctx():\\n        return sdpa_kernel(SDPBackend.EFFICIENT_ATTENTION)\\nexcept ImportError:  # older torch versions\\n    @contextlib.contextmanager\\n    def _sdpa_ctx():\\n        with torch.backends.cuda.sdp_kernel(\\n            enable_flash=False, enable_math=False, enable_mem_efficient=True\\n        ):\\n            yield\\n\\n# \\u2500\\u2500 DDP initialisation \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n# IMPORTANT: allocator config + device selection must happen BEFORE\\n# init_process_group. NCCL init creates each process's CUDA context, and the\\n# caching allocator reads PYTORCH_CUDA_ALLOC_CONF lazily on first CUDA call \\u2014\\n# setting it after init_process_group is too late, it's silently ignored.\\nLOCAL_RANK = int(os.environ[\\\"LOCAL_RANK\\\"])\\nos.environ[\\\"PYTORCH_CUDA_ALLOC_CONF\\\"] = \\\"expandable_segments:True\\\"\\ntorch.cuda.set_device(LOCAL_RANK)\\ntorch.backends.cuda.matmul.allow_tf32 = True\\ntorch.backends.cudnn.allow_tf32       = True\\n\\ndist.init_process_group(backend=\\\"nccl\\\")\\nWORLD_SIZE = dist.get_world_size()\\nIS_MASTER  = (LOCAL_RANK == 0)\\ndevice = f\\\"cuda:{LOCAL_RANK}\\\"\\n\\nif IS_MASTER:\\n    print(f\\\"DDP ready | world_size={WORLD_SIZE}\\\")\\n    for i in range(WORLD_SIZE):\\n        mem = torch.cuda.get_device_properties(i).total_memory / 1e9\\n        print(f\\\"  GPU {i}: {torch.cuda.get_device_name(i)}  {mem:.1f} GB\\\")\\n\\n# \\u2500\\u2500 Paths \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\nCKPT_DIR  = \\\"/kaggle/working/GPT500M/Checkpoints\\\"\\nCACHE_DIR = \\\"/tmp/HF_Cache\\\"\\n\\n# \\u2500\\u2500 Kaggle Model registry push target \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n# Set by the notebook (Section 3) via env var so checkpoints survive past this\\n# session without relying on /kaggle/working disk or the notebook \\\"commit\\\".\\nKAGGLE_MODEL_HANDLE = os.environ.get(\\\"KAGGLE_MODEL_HANDLE\\\", \\\"\\\")\\n\\n\\n# \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n# CONFIG\\n# \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n@dataclass\\nclass GPTConfig:\\n    # Architecture\\n    vocab_size : int   = 50257\\n    block_size : int   = 1024\\n    n_layer    : int   = 28\\n    n_head     : int   = 16\\n    n_embd     : int   = 1152\\n    dropout    : float = 0.0\\n\\n    # Training\\n    # 2xT4: micro_batch=1 per GPU, grad_accum=128\\n    # effective batch = 2 GPUs x 1 x 128 x 1024 = 262,144 tokens/step\\n    # (micro_batch dropped from 2->1 and grad_accum doubled to keep the same\\n    #  effective batch size while cutting per-step activation/allocator peak)\\n    num_epochs        : int   = 2\\n    micro_batch_size  : int   = 1\\n    grad_accum_steps  : int   = 128\\n    lr                : float = 3e-4\\n    weight_decay      : float = 0.1\\n    grad_clip         : float = 1.0\\n    warmup_steps      : int   = 1000\\n    train_steps       : int   = 112000\\n    eval_interval     : int   = 100\\n    eval_batches      : int   = 50\\n    tokenizer_name    : str   = \\\"gpt2\\\"\\n    max_checkpoints   : int   = 1\\n    device            : str   = \\\"cuda\\\"\\n\\ncfg = GPTConfig()\\ncfg.device = device\\n\\n\\n# \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n# MODEL\\n# \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\nclass CausalSelfAttention(nn.Module):\\n    def __init__(self, cfg: GPTConfig):\\n        super().__init__()\\n        assert cfg.n_embd % cfg.n_head == 0\\n        self.n_head   = cfg.n_head\\n        self.n_embd   = cfg.n_embd\\n        self.head_dim = cfg.n_embd // cfg.n_head\\n        self.dropout  = cfg.dropout\\n        self.c_attn     = nn.Linear(cfg.n_embd, 3 * cfg.n_embd, bias=False)\\n        self.c_proj     = nn.Linear(cfg.n_embd, cfg.n_embd,     bias=False)\\n        self.resid_drop = nn.Dropout(cfg.dropout)\\n        self._cache_k: Optional[torch.Tensor] = None\\n        self._cache_v: Optional[torch.Tensor] = None\\n\\n    def forward(self, x: torch.Tensor, use_cache: bool = False) -> torch.Tensor:\\n        B, T, C = x.shape\\n        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)\\n        def split_heads(t):\\n            return t.view(B, T, self.n_head, self.head_dim).transpose(1, 2)\\n        q, k, v = split_heads(q), split_heads(k), split_heads(v)\\n        if use_cache:\\n            if self._cache_k is not None:\\n                k = torch.cat([self._cache_k, k], dim=2)\\n                v = torch.cat([self._cache_v, v], dim=2)\\n            self._cache_k = k\\n            self._cache_v = v\\n        dropout_p = self.dropout if self.training else 0.0\\n        with _sdpa_ctx():\\n            y = F.scaled_dot_product_attention(q, k, v, attn_mask=None,\\n                                               dropout_p=dropout_p, is_causal=True)\\n        y = y.transpose(1, 2).contiguous().view(B, T, C)\\n        return self.resid_drop(self.c_proj(y))\\n\\n    def clear_cache(self):\\n        self._cache_k = None\\n        self._cache_v = None\\n\\n\\nclass FeedForward(nn.Module):\\n    def __init__(self, cfg: GPTConfig):\\n        super().__init__()\\n        self.net = nn.Sequential(\\n            nn.Linear(cfg.n_embd, 4 * cfg.n_embd, bias=False),\\n            nn.GELU(),\\n            nn.Linear(4 * cfg.n_embd, cfg.n_embd, bias=False),\\n            nn.Dropout(cfg.dropout),\\n        )\\n    def forward(self, x): return self.net(x)\\n\\n\\nclass TransformerBlock(nn.Module):\\n    def __init__(self, cfg: GPTConfig):\\n        super().__init__()\\n        self.ln1  = nn.LayerNorm(cfg.n_embd)\\n        self.attn = CausalSelfAttention(cfg)\\n        self.ln2  = nn.LayerNorm(cfg.n_embd)\\n        self.ffn  = FeedForward(cfg)\\n\\n    def _block_fn(self, x):\\n        x = x + self.attn(self.ln1(x), use_cache=False)\\n        x = x + self.ffn(self.ln2(x))\\n        return x\\n\\n    def forward(self, x, use_cache=False):\\n        if self.training and not use_cache:\\n            return grad_ckpt.checkpoint(self._block_fn, x, use_reentrant=False)\\n        x = x + self.attn(self.ln1(x), use_cache=use_cache)\\n        x = x + self.ffn(self.ln2(x))\\n        return x\\n\\n\\nclass GPT500M(nn.Module):\\n    def __init__(self, cfg: GPTConfig):\\n        super().__init__()\\n        self.cfg      = cfg\\n        self.tok_emb  = nn.Embedding(cfg.vocab_size, cfg.n_embd)\\n        self.pos_emb  = nn.Embedding(cfg.block_size, cfg.n_embd)\\n        self.drop     = nn.Dropout(cfg.dropout)\\n        self.blocks   = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layer)])\\n        self.ln_final = nn.LayerNorm(cfg.n_embd)\\n        self.head     = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)\\n        self.head.weight = self.tok_emb.weight\\n        self._init_weights()\\n        if IS_MASTER:\\n            n = self.num_params()\\n            print(f\\\"GPT-500M | {n:,} params ({n/1e6:.1f}M)\\\")\\n\\n    def _init_weights(self):\\n        for name, module in self.named_modules():\\n            if isinstance(module, nn.Linear):\\n                std = 0.02\\n                if name.endswith((\\\"c_proj\\\", \\\"net.2\\\")):\\n                    std = 0.02 / math.sqrt(2 * self.cfg.n_layer)\\n                nn.init.normal_(module.weight, mean=0.0, std=std)\\n                if module.bias is not None:\\n                    nn.init.zeros_(module.bias)\\n            elif isinstance(module, nn.Embedding):\\n                nn.init.normal_(module.weight, mean=0.0, std=0.02)\\n            elif isinstance(module, nn.LayerNorm):\\n                nn.init.ones_(module.weight)\\n                nn.init.zeros_(module.bias)\\n\\n    def num_params(self):\\n        return sum(p.numel() for n, p in self.named_parameters() if n != \\\"head.weight\\\")\\n\\n    def forward(self, idx, targets=None, use_cache=False):\\n        B, T = idx.shape\\n        assert T <= self.cfg.block_size\\n        positions = torch.arange(T, device=idx.device)\\n        x = self.drop(self.tok_emb(idx) + self.pos_emb(positions))\\n        for block in self.blocks:\\n            x = block(x, use_cache=use_cache)\\n        x      = self.ln_final(x)\\n        logits = self.head(x)\\n        loss = None\\n        if targets is not None:\\n            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))\\n        return logits, loss\\n\\n    def clear_kv_cache(self):\\n        for block in self.blocks:\\n            block.attn.clear_cache()\\n\\n\\n# \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n# CHECKPOINT UTILITIES\\n# \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\ndef _push_checkpoint_worker(tmp_dir, step):\\n    \\\"\\\"\\\"Runs in a background thread \\u2014 uploads a staged checkpoint copy to the\\n    Kaggle Model registry as a new version. Deletes the staged copy when done.\\\"\\\"\\\"\\n    try:\\n        result = subprocess.run(\\n            [\\\"kaggle\\\", \\\"models\\\", \\\"instances\\\", \\\"versions\\\", \\\"create\\\",\\n             KAGGLE_MODEL_HANDLE, \\\"-p\\\", tmp_dir, \\\"--notes\\\", f\\\"step {step}\\\"],\\n            capture_output=True, text=True, timeout=3600,\\n        )\\n        if result.returncode == 0:\\n            print(f\\\"  [kaggle-push] step {step} checkpoint uploaded -> {KAGGLE_MODEL_HANDLE}\\\")\\n        else:\\n            print(f\\\"  [kaggle-push] WARNING: upload failed for step {step}:\\\\n{result.stderr[-800:]}\\\")\\n    except Exception as e:\\n        print(f\\\"  [kaggle-push] WARNING: upload failed for step {step}: {e}\\\")\\n    finally:\\n        shutil.rmtree(tmp_dir, ignore_errors=True)\\n\\n\\ndef push_checkpoint_async(ckpt_path, step):\\n    \\\"\\\"\\\"\\n    Fire-and-forget upload of a checkpoint to the Kaggle Model registry so it\\n    survives independently of /kaggle/working and this session's lifetime.\\n    The file is copied to a throwaway staging dir *synchronously* first, so\\n    later checkpoint rotation (which deletes old .pt files in CKPT_DIR) can\\n    never race with an in-flight upload.\\n    \\\"\\\"\\\"\\n    if not KAGGLE_MODEL_HANDLE:\\n        print(\\\"  [kaggle-push] KAGGLE_MODEL_HANDLE not set \\u2014 skipping registry push.\\\")\\n        return\\n    tmp_dir = tempfile.mkdtemp()\\n    try:\\n        shutil.copy(ckpt_path, tmp_dir)\\n    except Exception as e:\\n        print(f\\\"  [kaggle-push] WARNING: could not stage checkpoint for upload: {e}\\\")\\n        shutil.rmtree(tmp_dir, ignore_errors=True)\\n        return\\n    threading.Thread(target=_push_checkpoint_worker, args=(tmp_dir, step), daemon=True).start()\\n\\n\\ndef save_checkpoint(model, optimizer, scaler, step, epoch, epoch_step, losses, cfg):\\n    \\\"\\\"\\\"\\n    Rank 0 only. Strips DDP module. prefix so the .pt is single-GPU compatible.\\n    \\\"\\\"\\\"\\n    raw_sd    = model.module.state_dict()\\n    ckpt_path = os.path.join(CKPT_DIR, f\\\"ckpt_{step:06d}.pt\\\")\\n    torch.save({\\n        \\\"step\\\"       : step,\\n        \\\"epoch\\\"      : epoch,\\n        \\\"epoch_step\\\" : epoch_step,\\n        \\\"model\\\"      : raw_sd,\\n        \\\"optimizer\\\"  : optimizer.state_dict(),\\n        \\\"scaler\\\"     : scaler.state_dict(),\\n        \\\"cfg\\\"        : cfg,\\n        \\\"losses\\\"     : losses,\\n    }, ckpt_path)\\n\\n    train_loss = losses[\\\"train\\\"]\\n    val_loss   = losses[\\\"val\\\"]\\n    print(f\\\"  checkpoint saved -> {ckpt_path}  \\\"\\n          f\\\"(epoch {epoch} | step {step} | train {train_loss:.4f} | val {val_loss:.4f})\\\")\\n\\n    # Push a copy to the Kaggle Model registry (async, non-blocking) so it\\n    # persists independently of /kaggle/working and this session.\\n    push_checkpoint_async(ckpt_path, step)\\n\\n    # Rotate \\u2014 keep only max_checkpoints most recent\\n    all_ckpts = sorted(glob.glob(os.path.join(CKPT_DIR, \\\"ckpt_*.pt\\\")),\\n                       key=os.path.getmtime)\\n    for old in all_ckpts[: max(0, len(all_ckpts) - cfg.max_checkpoints)]:\\n        os.remove(old)\\n        print(f\\\"  deleted old checkpoint: {old}\\\")\\n\\n\\ndef load_checkpoint(path, raw_model, optimizer, scaler):\\n    \\\"\\\"\\\"\\n    Works for both single-GPU and DDP-saved checkpoints.\\n    raw_model must be the unwrapped GPT500M (before DDP wrapping).\\n    \\\"\\\"\\\"\\n    print(f\\\"Loading checkpoint: {path}\\\")\\n    ckpt = torch.load(path, map_location=\\\"cpu\\\", weights_only=False)\\n\\n    cfg        = ckpt[\\\"cfg\\\"]\\n    cfg.device = device\\n\\n    # Strip module. prefix (safety net \\u2014 save_checkpoint already does this)\\n    sd = {k.replace(\\\"module.\\\", \\\"\\\"): v for k, v in ckpt[\\\"model\\\"].items()}\\n    raw_model.load_state_dict(sd)\\n    del sd, ckpt[\\\"model\\\"]\\n\\n    optimizer.load_state_dict(ckpt[\\\"optimizer\\\"])\\n    for state in optimizer.state.values():\\n        for k, v in state.items():\\n            if isinstance(v, torch.Tensor):\\n                state[k] = v.to(device)\\n    del ckpt[\\\"optimizer\\\"]\\n\\n    if \\\"scaler\\\" in ckpt:\\n        scaler.load_state_dict(ckpt[\\\"scaler\\\"])\\n\\n    gc.collect()\\n    torch.cuda.empty_cache()\\n\\n    step       = ckpt[\\\"step\\\"]\\n    epoch      = ckpt.get(\\\"epoch\\\",      0)\\n    epoch_step = ckpt.get(\\\"epoch_step\\\", 0)\\n    losses     = ckpt[\\\"losses\\\"]\\n\\n    if IS_MASTER:\\n        train_loss = losses[\\\"train\\\"]\\n        val_loss   = losses[\\\"val\\\"]\\n        allocated  = torch.cuda.memory_allocated() / 1e9\\n        reserved   = torch.cuda.memory_reserved()  / 1e9\\n        print(f\\\"  resumed \\u2014 global step {step} | epoch {epoch} | epoch_step {epoch_step}\\\")\\n        print(f\\\"  train loss {train_loss:.4f} | val loss {val_loss:.4f}\\\")\\n        print(f\\\"  VRAM after load: {allocated:.2f} GB allocated / {reserved:.2f} GB reserved\\\")\\n\\n    return cfg, step, epoch, epoch_step, losses\\n\\n\\n# \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n# DATA PIPELINE\\n# \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\nimport tiktoken\\nfrom datasets import load_dataset\\n\\nenc = tiktoken.get_encoding(\\\"gpt2\\\")\\nEOT = enc.eot_token\\n\\n\\nclass TokenBuffer:\\n    \\\"\\\"\\\"\\n    Each DDP rank gets its own TokenBuffer with a different seed offset,\\n    so the two GPUs never process the same documents in the same order.\\n    \\\"\\\"\\\"\\n    def __init__(self, iterator, block_size: int):\\n        self._iter       = iterator\\n        self._block_size = block_size\\n        self._buf: list  = []\\n\\n    def _generate(self) -> Iterator:\\n        for doc in self._iter:\\n            tokens = enc.encode_ordinary(doc[\\\"text\\\"]) + [EOT]\\n            self._buf.extend(tokens)\\n            while len(self._buf) >= self._block_size + 1:\\n                chunk = self._buf[: self._block_size + 1]\\n                self._buf = self._buf[self._block_size + 1 :]\\n                yield chunk\\n\\n\\ndef make_stream(epoch: int, skip_docs: int = 0):\\n    ds = load_dataset(\\n        \\\"open-web-math/open-web-math\\\",\\n        split     = \\\"train\\\",\\n        streaming = True,\\n        cache_dir = CACHE_DIR,\\n    )\\n    # Each rank shuffles with a different seed so they see different docs\\n    ds = ds.shuffle(seed=42 + epoch + LOCAL_RANK * 1000, buffer_size=10_000)\\n    if skip_docs > 0 and IS_MASTER:\\n        print(f\\\"  fast-forwarding {skip_docs:,} documents in epoch {epoch} ...\\\")\\n    if skip_docs > 0:\\n        ds = ds.skip(skip_docs)\\n    return iter(ds)\\n\\n\\ndef make_val_tensors(n_val_docs: int = 500, block_size: int = 1024):\\n    \\\"\\\"\\\"Only rank 0 builds and holds the validation set.\\\"\\\"\\\"\\n    if IS_MASTER:\\n        print(f\\\"Building validation set from {n_val_docs} val docs ...\\\")\\n    val_ds = load_dataset(\\n        \\\"open-web-math/open-web-math\\\",\\n        split     = \\\"train\\\",\\n        streaming = True,\\n        cache_dir = CACHE_DIR,\\n    )\\n    val_tokens = []\\n    for i, doc in enumerate(val_ds):\\n        if i >= n_val_docs:\\n            break\\n        val_tokens.extend(enc.encode_ordinary(doc[\\\"text\\\"]) + [EOT])\\n    n = (len(val_tokens) // (block_size + 1)) * (block_size + 1)\\n    return torch.tensor(val_tokens[:n], dtype=torch.int64)\\n\\n\\n# \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n# TRAINING UTILITIES\\n# \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\ndef get_lr(step: int, cfg: GPTConfig) -> float:\\n    if step < cfg.warmup_steps:\\n        return cfg.lr * (step + 1) / cfg.warmup_steps\\n    progress = (step - cfg.warmup_steps) / max(1, cfg.train_steps - cfg.warmup_steps)\\n    return cfg.lr * (0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * progress)))\\n\\n\\n@torch.no_grad()\\ndef estimate_loss(model, val_data, cfg):\\n    \\\"\\\"\\\"\\n    Rank 0 runs eval; result is broadcast to all ranks via dist.broadcast.\\n    All ranks must call this function (for the broadcast to work).\\n    \\\"\\\"\\\"\\n    loss_tensor = torch.zeros(1, device=device)\\n    if IS_MASTER:\\n        model.eval()\\n        val_gpu   = val_data.to(device)\\n        max_start = len(val_gpu) - cfg.block_size - 1\\n        losses = []\\n        with torch.amp.autocast(\\\"cuda\\\"):\\n            for _ in range(cfg.eval_batches):\\n                ix = torch.randint(max_start, (cfg.micro_batch_size,))\\n                x  = torch.stack([val_gpu[i     : i + cfg.block_size    ] for i in ix])\\n                y  = torch.stack([val_gpu[i + 1 : i + cfg.block_size + 1] for i in ix])\\n                _, loss = model(x, y)\\n                losses.append(loss.item())\\n        del val_gpu\\n        torch.cuda.empty_cache()\\n        loss_tensor[0] = sum(losses) / len(losses)\\n        model.train()\\n    dist.broadcast(loss_tensor, src=0)\\n    return {\\\"val\\\": loss_tensor.item()}\\n\\n\\n# \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n# BUILD MODEL, OPTIMIZER, SCALER\\n# \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\ngc.collect()\\ntorch.cuda.empty_cache()\\n\\nraw_model = GPT500M(cfg)\\n\\ndecay_params    = [p for n, p in raw_model.named_parameters() if p.dim() >= 2 and p.requires_grad]\\nno_decay_params = [p for n, p in raw_model.named_parameters() if p.dim() <  2 and p.requires_grad]\\n\\n# 8-bit optimizer state cuts AdamW's exp_avg/exp_avg_sq footprint from ~4GB to\\n# ~1GB on a 505M-param model. Prefer the *paged* variant: it spills state to\\n# CPU pinned memory under GPU pressure instead of raising OOM, which gives a\\n# real safety margin on a card this close to the edge. Falls back to plain\\n# AdamW8bit, then to standard fp32 AdamW if bitsandbytes isn't installed.\\ntry:\\n    import bitsandbytes as bnb\\n    OptClass = getattr(bnb.optim, \\\"PagedAdamW8bit\\\", None) or bnb.optim.AdamW8bit\\n    optimizer = OptClass([\\n        {\\\"params\\\": decay_params,    \\\"weight_decay\\\": cfg.weight_decay},\\n        {\\\"params\\\": no_decay_params, \\\"weight_decay\\\": 0.0},\\n    ], lr=cfg.lr, betas=(0.9, 0.95))\\n    if IS_MASTER:\\n        print(f\\\"Optimizer: bitsandbytes {OptClass.__name__} (reduced VRAM optimizer state)\\\")\\nexcept ImportError:\\n    optimizer = torch.optim.AdamW([\\n        {\\\"params\\\": decay_params,    \\\"weight_decay\\\": cfg.weight_decay},\\n        {\\\"params\\\": no_decay_params, \\\"weight_decay\\\": 0.0},\\n    ], lr=cfg.lr, betas=(0.9, 0.95), fused=True)\\n    if IS_MASTER:\\n        print(\\\"Optimizer: fp32 AdamW (bitsandbytes not found \\u2014 pip install bitsandbytes to save VRAM)\\\")\\n\\nscaler = torch.amp.GradScaler(\\\"cuda\\\")\\n\\ndef _log_vram(label):\\n    \\\"\\\"\\\"Per-rank VRAM snapshot \\u2014 printed from every rank, since OOM can hit\\n    either GPU independently and rank-0-only logging hides that.\\\"\\\"\\\"\\n    a = torch.cuda.memory_allocated(device) / 1e9\\n    r = torch.cuda.memory_reserved(device)  / 1e9\\n    print(f\\\"  [vram][rank{LOCAL_RANK}] {label}: allocated {a:.2f}GB reserved {r:.2f}GB\\\")\\n\\n# \\u2500\\u2500 Resume or fresh start \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\ncheckpoints = sorted(glob.glob(os.path.join(CKPT_DIR, \\\"ckpt_*.pt\\\")), key=os.path.getmtime)\\nlatest_ckpt = checkpoints[-1] if checkpoints else None\\n\\nraw_model.to(device)\\n_log_vram(\\\"after model.to(device)\\\")\\n\\nif latest_ckpt:\\n    cfg, start_step, start_epoch, start_epoch_step, loaded_losses = \\\\\\n        load_checkpoint(latest_ckpt, raw_model, optimizer, scaler)\\nelse:\\n    if IS_MASTER:\\n        print(\\\"No checkpoint found \\u2014 starting from scratch.\\\")\\n    start_step       = 0\\n    start_epoch      = 0\\n    start_epoch_step = 0\\n    loaded_losses    = {\\\"train\\\": float(\\\"nan\\\"), \\\"val\\\": float(\\\"nan\\\")}\\n_log_vram(\\\"after checkpoint resume / fresh init\\\")\\n\\n# Wrap in DDP AFTER loading weights\\nmodel = DDP(raw_model, device_ids=[LOCAL_RANK], find_unused_parameters=False)\\ntorch.cuda.empty_cache()\\n_log_vram(\\\"after DDP wrap\\\")\\n\\n# \\u2500\\u2500 Step & epoch counters \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\nDOCS_PER_EPOCH  = 6_300_000\\nSTEPS_PER_EPOCH = DOCS_PER_EPOCH * 350 // (cfg.block_size + 1) // cfg.grad_accum_steps\\ncfg.train_steps = STEPS_PER_EPOCH * cfg.num_epochs\\n\\nif IS_MASTER:\\n    tokens_per_step = WORLD_SIZE * cfg.micro_batch_size * cfg.grad_accum_steps * cfg.block_size\\n    print(f\\\"\\\\nSteps per epoch       : {STEPS_PER_EPOCH:,}\\\")\\n    print(f\\\"Total steps           : {cfg.train_steps:,} ({cfg.num_epochs} epoch(s))\\\")\\n    print(f\\\"Start step            : {start_step}\\\")\\n    print(f\\\"Effective tokens/step : {tokens_per_step:,}\\\")\\n\\n\\n# \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n# BUILD VAL SET (rank 0 only) & DATA STREAM\\n# \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\nval_data = make_val_tensors(n_val_docs=500, block_size=cfg.block_size) if IS_MASTER else None\\nif IS_MASTER:\\n    print(f\\\"Val set ready \\u2014 {len(val_data):,} tokens\\\")\\n\\ntrain_stream_iter = make_stream(\\n    epoch     = start_epoch,\\n    skip_docs = start_epoch_step * cfg.grad_accum_steps * cfg.micro_batch_size,\\n)\\ntrain_buf = TokenBuffer(train_stream_iter, block_size=cfg.block_size)\\nbuf_gen   = train_buf._generate()\\n\\n\\n# \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n# TRAINING LOOP\\n# \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\nmodel.train()\\ntrain_loss_acc = 0.0\\nt0 = time.time()\\n\\nepoch      = start_epoch\\nepoch_step = start_epoch_step\\n\\nfor step in range(start_step, cfg.train_steps):\\n\\n    # Epoch boundary\\n    if epoch_step >= STEPS_PER_EPOCH:\\n        epoch      += 1\\n        epoch_step  = 0\\n        if epoch >= cfg.num_epochs:\\n            if IS_MASTER:\\n                print(f\\\"\\\\nCompleted {cfg.num_epochs} epoch(s). Training done.\\\")\\n            break\\n        if IS_MASTER:\\n            sep = \\\"=\\\" * 60\\n            print(f\\\"\\\\n{sep}\\\\nStarting epoch {epoch}\\\\n{sep}\\\")\\n        train_stream_iter = make_stream(epoch=epoch, skip_docs=0)\\n        train_buf = TokenBuffer(train_stream_iter, block_size=cfg.block_size)\\n        buf_gen   = train_buf._generate()\\n\\n    # LR schedule\\n    lr = get_lr(step, cfg)\\n    for pg in optimizer.param_groups:\\n        pg[\\\"lr\\\"] = lr\\n\\n    optimizer.zero_grad(set_to_none=True)\\n    accum_loss = 0.0\\n\\n    with torch.amp.autocast(\\\"cuda\\\"):\\n        for micro_step in range(cfg.grad_accum_steps):\\n            try:\\n                chunks = [next(buf_gen) for _ in range(cfg.micro_batch_size)]\\n            except StopIteration:\\n                epoch_step = STEPS_PER_EPOCH\\n                break\\n\\n            data = torch.tensor(chunks, dtype=torch.int64)\\n            x = data[:, :-1].to(device)\\n            y = data[:, 1: ].to(device)\\n\\n            # Only sync gradients on the last micro-step\\n            is_last_micro = (micro_step == cfg.grad_accum_steps - 1)\\n            ctx = model.no_sync() if not is_last_micro else contextlib.nullcontext()\\n            with ctx:\\n                _, loss = model(x, y)\\n                loss = loss / cfg.grad_accum_steps\\n                scaler.scale(loss).backward()\\n            accum_loss += loss.item()\\n\\n    scaler.unscale_(optimizer)\\n    torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)\\n    scaler.step(optimizer)\\n    scaler.update()\\n\\n    epoch_step     += 1\\n    train_loss_acc += accum_loss\\n\\n    if step == start_step:\\n        _log_vram(\\\"after first optimizer.step()\\\")\\n\\n    if step % 50 == 0:\\n        torch.cuda.empty_cache()\\n        if step != start_step:\\n            _log_vram(f\\\"step {step}\\\")\\n\\n    if IS_MASTER and step % 10 == 0:\\n        smooth_loss = train_loss_acc / max(1, step - start_step + 1)\\n        print(f\\\"ep {epoch} | step {step:>6d}/{cfg.train_steps} | \\\"\\n              f\\\"loss {accum_loss:.4f} | smooth {smooth_loss:.4f} | lr {lr:.2e}\\\", end=\\\"\\\\r\\\")\\n\\n    if (step % cfg.eval_interval == 0) or (step == cfg.train_steps - 1):\\n        # All ranks participate (dist.broadcast inside estimate_loss)\\n        val_losses   = estimate_loss(model, val_data, cfg)\\n        smooth_train = train_loss_acc / max(1, step - start_step + 1)\\n        losses       = {\\\"train\\\": smooth_train, \\\"val\\\": val_losses[\\\"val\\\"]}\\n\\n        if IS_MASTER:\\n            elapsed = time.time() - t0\\n            sep = \\\"-\\\" * 65\\n            print(f\\\"\\\\n{sep}\\\")\\n            print(f\\\"EVAL  ep {epoch} | step {step:>6d} | \\\"\\n                  f\\\"train {losses['train']:.4f} | val {losses['val']:.4f} | {elapsed:.0f}s\\\")\\n            save_checkpoint(model, optimizer, scaler,\\n                            step, epoch, epoch_step, losses, cfg)\\n            t0 = time.time()\\n\\n        # All ranks sync before continuing\\n        dist.barrier()\\n        model.train()\\n\\nif IS_MASTER:\\n    print(\\\"\\\\nTraining complete.\\\")\\n\\ndist.destroy_process_group()\\n\"")
with open(SCRIPT, 'w') as _f:
    _f.write(_script)
print(f"Training script written to {SCRIPT}")
print(f"Lines : {len(_script.splitlines())}")

## 5. Verify Script

In [ ]:
import ast
with open(SCRIPT) as f:
    src = f.read()
try:
    ast.parse(src)
    print("✓ Script syntax OK")
    print(f"  Lines : {len(src.splitlines())}")
    print(f"  Path  : {SCRIPT}")
except SyntaxError as e:
    print(f"✗ Syntax error: {e}")

## 6. Launch Training

`torchrun --nproc_per_node=2` spawns 2 processes, one per GPU.  
Rank 0 handles all printing — output will not be duplicated.  

**To resume:** just re-run this cell. The script auto-detects the latest checkpoint.

In [ ]:
!torchrun --nproc_per_node=2 --master_port=29500 {SCRIPT}

## 7. Inspect Checkpoint

Loads the latest checkpoint metadata for a quick sanity check.  
Paste the model class here and uncomment the generation block to run inference.

In [ ]:
import os, glob, torch

CKPT_DIR = '/kaggle/working/GPT500M/Checkpoints'
checkpoints = sorted(glob.glob(os.path.join(CKPT_DIR, 'ckpt_*.pt')), key=os.path.getmtime)
assert checkpoints, "No checkpoints found — run the training cell first."

latest = checkpoints[-1]
print(f"Loading: {latest}")
ckpt = torch.load(latest, map_location='cpu', weights_only=False)

step       = ckpt['step']
epoch      = ckpt.get('epoch', 0)
train_loss = ckpt['losses']['train']
val_loss   = ckpt['losses']['val']
print(f"Step       : {step}")
print(f"Epoch      : {epoch}")
print(f"Train loss : {train_loss:.4f}")
print(f"Val loss   : {val_loss:.4f}")

# ── Generation (uncomment after pasting model classes above) ─────────────────
# import tiktoken, math
# from train_ddp import GPT500M   # or paste the class here
# device    = 'cuda:0'
# enc       = tiktoken.get_encoding('gpt2')
# gen_cfg   = ckpt['cfg']
# gen_model = GPT500M(gen_cfg).to(device)
# sd = {k.replace('module.', ''): v for k, v in ckpt['model'].items()}
# gen_model.load_state_dict(sd)
# gen_model.eval()
#
# PROMPT         = "The derivative of sin(x) with respect to x is"
# MAX_NEW_TOKENS = 256
# TEMPERATURE    = 0.8
# TOP_K          = 50
#
# ids = enc.encode_ordinary(PROMPT)
# x   = torch.tensor([ids], dtype=torch.int64).to(device)
# with torch.no_grad():
#     for _ in range(MAX_NEW_TOKENS):
#         logits, _ = gen_model(x[:, -gen_cfg.block_size:])
#         logits = logits[:, -1, :] / TEMPERATURE
#         top_k_vals, top_k_idx = torch.topk(logits, TOP_K)
#         probs = torch.softmax(top_k_vals, dim=-1)
#         next_id = top_k_idx[0, torch.multinomial(probs[0], 1)]
#         x = torch.cat([x, next_id.unsqueeze(0).unsqueeze(0)], dim=1)
# print(enc.decode(x[0].tolist()))